# Trading Portfolio Analysis — Data Collection Pipeline

This notebook builds the full dataset from scratch:

1. Download daily adjusted close prices for 110 tickers (11 sectors x 10 tickers) from Yahoo Finance
2. Reshape from wide (one column per ticker) to long (one row per Date/Ticker) format
3. Attach each ticker's sector and compute daily returns
4. Download the SPY benchmark and compute its daily return
5. Load everything into a local SQLite database
6. Run sanity checks to confirm nothing was lost/corrupted along the way

Every path below is **relative to the project root**, so this notebook runs the same
on any machine — no hardcoded `C:\Users\...` paths.


In [ ]:
import pandas as pd
import yfinance as yf
import sqlite3
from pathlib import Path

# Project root = parent of this notebook's folder (notebooks/ -> project root)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_DIR = PROJECT_ROOT / "data"
SQL_DIR = PROJECT_ROOT / "sql"
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2020-01-01"
BENCHMARK_TICKER = "SPY"

print("Project root:", PROJECT_ROOT)


## Step 1 — Define the ticker universe

This is the 110-ticker, 11-sector universe from the existing project (10 tickers per sector,
GICS-style sector buckets). Keeping this as an explicit dict — rather than a bare list — means
the sector tag travels with the ticker instead of being bolted on later, which is where a lot
of the earlier version's confusion came from.

In [ ]:
TICKER_SECTORS = {
    'EA': 'Communication Services', 'CHTR': 'Communication Services', 'META': 'Communication Services',
    'NFLX': 'Communication Services', 'DIS': 'Communication Services', 'T': 'Communication Services',
    'CMCSA': 'Communication Services', 'TMUS': 'Communication Services', 'VZ': 'Communication Services',
    'GOOGL': 'Communication Services',
    'AMZN': 'Consumer Discretionary', 'SBUX': 'Consumer Discretionary', 'ROST': 'Consumer Discretionary',
    'MCD': 'Consumer Discretionary', 'NKE': 'Consumer Discretionary', 'TSLA': 'Consumer Discretionary',
    'LOW': 'Consumer Discretionary', 'MAR': 'Consumer Discretionary', 'TJX': 'Consumer Discretionary',
    'HD': 'Consumer Discretionary',
    'KO': 'Consumer Staples', 'KR': 'Consumer Staples', 'COST': 'Consumer Staples', 'KMB': 'Consumer Staples',
    'MDLZ': 'Consumer Staples', 'PEP': 'Consumer Staples', 'PG': 'Consumer Staples', 'WMT': 'Consumer Staples',
    'CL': 'Consumer Staples', 'MO': 'Consumer Staples',
    'EOG': 'Energy', 'MPC': 'Energy', 'PSX': 'Energy', 'HAL': 'Energy', 'SLB': 'Energy', 'XOM': 'Energy',
    'CVX': 'Energy', 'COP': 'Energy', 'BKR': 'Energy', 'DVN': 'Energy',
    'AXP': 'Financials', 'SCHW': 'Financials', 'C': 'Financials', 'JPM': 'Financials', 'BLK': 'Financials',
    'GS': 'Financials', 'MS': 'Financials', 'BAC': 'Financials', 'USB': 'Financials', 'WFC': 'Financials',
    'MRK': 'Health Care', 'JNJ': 'Health Care', 'ABT': 'Health Care', 'TMO': 'Health Care', 'UNH': 'Health Care',
    'AMGN': 'Health Care', 'LLY': 'Health Care', 'PFE': 'Health Care', 'ABBV': 'Health Care', 'DHR': 'Health Care',
    'RTX': 'Industrials', 'BA': 'Industrials', 'MMM': 'Industrials', 'CAT': 'Industrials', 'LMT': 'Industrials',
    'EMR': 'Industrials', 'DE': 'Industrials', 'UPS': 'Industrials', 'GE': 'Industrials', 'FDX': 'Industrials',
    'CRM': 'Information Technology', 'AVGO': 'Information Technology', 'ORCL': 'Information Technology',
    'NVDA': 'Information Technology', 'AAPL': 'Information Technology', 'MSFT': 'Information Technology',
    'IBM': 'Information Technology', 'ADBE': 'Information Technology', 'AMD': 'Information Technology',
    'CSCO': 'Information Technology',
    'VMC': 'Materials', 'SHW': 'Materials', 'APD': 'Materials', 'ECL': 'Materials', 'FCX': 'Materials',
    'IP': 'Materials', 'NEM': 'Materials', 'MLM': 'Materials', 'PKG': 'Materials', 'LIN': 'Materials',
    'EQR': 'Real Estate', 'EQIX': 'Real Estate', 'WELL': 'Real Estate', 'PLD': 'Real Estate', 'DLR': 'Real Estate',
    'AMT': 'Real Estate', 'CCI': 'Real Estate', 'O': 'Real Estate', 'AVB': 'Real Estate', 'SPG': 'Real Estate',
    'ED': 'Utilities', 'DUK': 'Utilities', 'D': 'Utilities', 'AEP': 'Utilities', 'XEL': 'Utilities',
    'SRE': 'Utilities', 'SO': 'Utilities', 'PEG': 'Utilities', 'EXC': 'Utilities', 'NEE': 'Utilities',
}

TICKERS = sorted(TICKER_SECTORS.keys())
print(f"{len(TICKERS)} tickers across {len(set(TICKER_SECTORS.values()))} sectors")


## Step 2 — Download raw prices

`yf.download` with a list of tickers returns a **wide** DataFrame with a MultiIndex column
(`(field, ticker)`). We only need the adjusted close, so we pull that one field out immediately
rather than carrying Open/High/Low/Volume around for the rest of the pipeline.

Note: `auto_adjust=True` (the current yfinance default) makes the plain `Close` column already
split/dividend-adjusted — that's what earlier versions of this pipeline were calling "Adj Close".

In [ ]:
raw = yf.download(TICKERS, start=START_DATE, auto_adjust=True, progress=False)

# raw.columns is a MultiIndex like ('Close', 'AAPL'), ('Close', 'MSFT'), ...
close_wide = raw['Close'].copy()
close_wide.index.name = 'Date'

close_wide.to_csv(RAW_DIR / 'stock_data_raw.csv')
print(close_wide.shape)
close_wide.head()


## Step 3 — Reshape to long format + attach sector

Long format (`Date, Ticker, Adj_Close`) is what the SQL table and Power BI both expect —
one row per observation, not one column per ticker. This is the step that was missing a clean
home in the old notebook (it existed, but only for the 12-ticker version).

In [ ]:
long = (
    close_wide
    .reset_index()
    .melt(id_vars='Date', var_name='Ticker', value_name='Adj_Close')
    .dropna(subset=['Adj_Close'])
)

long['Sector'] = long['Ticker'].map(TICKER_SECTORS)
long = long.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print(long.shape)
long.head()


## Step 4 — Compute daily returns

Daily return per ticker, computed **within each ticker group** (never across the boundary
between two different tickers) using `groupby().pct_change()`. The first day for each ticker
is correctly left as `NaN` — there's no prior day to compare to.

In [ ]:
long['Daily_return'] = long.groupby('Ticker')['Adj_Close'].pct_change()

out_path = DATA_DIR / 'stock_data.csv'
long.to_csv(out_path, index=False)
print(f"Saved {len(long):,} rows to {out_path}")
long.head()


## Step 5 — Benchmark data (SPY)

Same treatment for the benchmark index, kept in its own file/table since it isn't part of the
110-ticker universe and shouldn't be joined in as if it were just another stock.

In [ ]:
bench_raw = yf.download(BENCHMARK_TICKER, start=START_DATE, auto_adjust=True, progress=False)
bench = bench_raw['Close'].reset_index()
bench.columns = ['Date', 'Adj_Close']
bench['Ticker'] = BENCHMARK_TICKER
bench['Sector'] = None
bench['Benchmark_return'] = bench['Adj_Close'].pct_change()
bench = bench[['Date', 'Ticker', 'Adj_Close', 'Sector', 'Benchmark_return']]

bench.to_csv(DATA_DIR / 'benchmark_data.csv', index=False)
print(f"Saved {len(bench):,} rows")
bench.head()


## Step 6 — Load into SQLite

We use the schema in `sql/schema.sql` (SQLite-compatible — no `dbo.` prefixes, `IDENTITY`, or
`NVARCHAR`, which are SQL Server-only syntax) rather than hand-rolling `CREATE TABLE` here, so
the schema lives in one place and can be inspected/edited without touching this notebook.

In [ ]:
db_path = DATA_DIR / 'stock_data.db'
conn = sqlite3.connect(db_path)

with open(SQL_DIR / 'schema.sql') as f:
    conn.executescript(f.read())

long.to_sql('stock_data', conn, if_exists='append', index=False)
bench.to_sql('benchmark_data', conn, if_exists='append', index=False)

sector_dim = (
    long[['Ticker', 'Sector']]
    .drop_duplicates()
    .assign(Date=long['Date'].min())
    [['Date', 'Ticker', 'Sector']]
)
sector_dim.to_sql('sector_dim', conn, if_exists='append', index=False)

conn.commit()
print("Loaded into", db_path)


## Step 7 — Sanity checks

Cheap checks that would have caught the earlier mismatches (12-ticker notebook vs. 110-ticker
CSV, partial SQLite load) immediately instead of silently sitting in the repo.

In [ ]:
checks = pd.read_sql('SELECT COUNT(*) AS rows, COUNT(DISTINCT Ticker) AS tickers FROM stock_data', conn)
print(checks)

assert checks['tickers'][0] == len(TICKERS), "Ticker count mismatch between source dict and loaded table!"
expected_min_rows = len(TICKERS) * 1000  # rough floor, adjust as history grows
assert checks['rows'][0] > expected_min_rows, "Row count looks too low — check the download step."

null_check = pd.read_sql(
    "SELECT Ticker, Date FROM stock_data WHERE Adj_Close IS NULL", conn
)
print(f"{len(null_check)} rows with null Adj_Close (expect a handful on the most recent trading day if data hasn't fully settled):")
print(null_check)

conn.close()
print("All checks passed.")
